In [ ]:
"""
VisualStep — ACJ-Score & Instruction Type Difficulty Taxonomy
==============================================================
Run from Google Colab after mounting Drive:

    from google.colab import drive
    drive.mount('/content/drive')

Place all CSVs in the same Drive folder and update BASE_DIR below.

Outputs:
  1. ACJ-Score weights (from regression against human ratings)
  2. Per-model ACJ-Score table
  3. Correlation table: BLEU-4 vs human, ACI% vs human, ACJ-Score vs human
  4. Instruction type difficulty taxonomy (ACI% per type × model)
"""

In [ ]:
"""
VisualStep — ACJ-Score & Instruction Type Difficulty Taxonomy  (v3)
=====================================================================
Run from Google Colab after mounting Drive:

    from google.colab import drive
    drive.mount('/content/drive')

Requires:
    pip install spacy
    python -m spacy download en_core_web_sm
"""

CELL 1 - Install dependancies

In [ ]:
!pip install spacy
!python -m spacy download en_core_web_sm
!pip install -q evaluate sacrebleu


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── CELL1: Human Evaluation × Automatic Metrics Correlation ──────────────────
import pandas as pd
import numpy as np
import json
from scipy import stats
import re
import importlib, evaluate

# ── 1. Load files ────────────────────────────────────────────────────────────
# Adjust paths to wherever you stored them in Drive
HUMAN_EVAL_PATH  = "/content/drive/MyDrive/VisualStep/evaluation/human_eval_50_final.csv"
VS_OUTPUTS_PATH  = "/content/drive/MyDrive/VisualStep/results/test_outputs_visualstep.csv"  # your main results CSV

human = pd.read_csv(HUMAN_EVAL_PATH, encoding='latin-1')
vs    = pd.read_csv(VS_OUTPUTS_PATH)

# ── 2. Exclude card 21 (JSON_ERROR) ─────────────────────────────────────────
human = human[human['card_num'] != 21].copy()

# ── 3. Compute mean human rating per card ───────────────────────────────────
human['mean_rating'] = (human['examiner1_rating'] + human['examiner2_rating']) / 2
# ── 3b. Normalise input key for robust merging ────
import unicodedata

def normalise(s):
    s = str(s)
    # curly/smart quotes → straight
    s = s.replace('\u2018', "'").replace('\u2019', "'")
    s = s.replace('\u201c', '"').replace('\u201d', '"')
    # windows-1252 curly apostrophe
    s = s.replace('\x92', "'").replace('\x91', "'")
    return ' '.join(s.split()).strip()

human['input_key'] = human['input'].apply(normalise)
vs['input_key']    = vs['input'].apply(normalise)

# ── 4. Compute per-card BLEU-4, ACI%, JSON% from VS outputs ─────────────────
bleu_metric = evaluate.load("sacrebleu")

def safe_parse(s):
    try:
        return json.loads(str(s))
    except Exception:
        return None

def count_steps(parsed):
    if parsed and 'steps' in parsed:
        return len(parsed['steps'])
    return 0

def bleu4(ref, hyp):
    """sacrebleu expects list[str] refs and str hyp."""
    try:
        result = bleu_metric.compute(predictions=[hyp], references=[[ref]])
        return result['score'] / 100.0   # 0–1
    except Exception:
        return np.nan

def aci_score(gold_parsed, pred_parsed):
    """Action-Count Integrity: |n_gold − n_pred| / n_gold, then 1 − that."""
    n_g = count_steps(gold_parsed)
    n_p = count_steps(pred_parsed)
    if n_g == 0:
        return np.nan
    return 1.0 - abs(n_g - n_p) / n_g

def json_ok(pred_str):
    return 1.0 if safe_parse(pred_str) is not None else 0.0

rows = []
for _, row in human.iterrows():
    card  = row['card_num']
    # Find matching row in VS outputs — match on 'input' column (adjust col name if needed)
    vs_row = vs[vs['input_key'] == row['input_key']]
    if vs_row.empty:
        # Try card_num if available
        vs_row = vs[vs.get('card_num', pd.Series(dtype=int)) == card] if 'card_num' in vs.columns else pd.DataFrame()
    if vs_row.empty:
        rows.append({'card_num': card, 'bleu4': np.nan, 'aci': np.nan,
                     'json_pct': np.nan, 'mean_rating': row['mean_rating']})
        continue

    vs_row     = vs_row.iloc[0]
    gold_str   = str(row['gold_output'])
    pred_str   = str(vs_row.get('model_output', vs_row.get('output', '')))
    gold_p     = safe_parse(gold_str)
    pred_p     = safe_parse(pred_str)

    rows.append({
        'card_num':    card,
        'bleu4':       bleu4(gold_str, pred_str),
        'aci':         aci_score(gold_p, pred_p),
        'json_pct':    json_ok(pred_str),
        'mean_rating': row['mean_rating'],
    })

df = pd.DataFrame(rows).dropna(subset=['mean_rating'])

# ── 5. Normalise BLEU-4 and ACI% to [0,1] then compute ACJ-Score ───────────
df['bleu4_norm'] = df['bleu4'] / df['bleu4'].max()   # already 0–1, but re-scale to observed max
df['acj_score'] = 0.50 * df['bleu4_norm'] + 0.40 * df['aci'] + 0.10 * df['json_pct']

# ── 6. Correlation: Pearson r and Spearman ρ ────────────────────────────────
metrics = {
    'BLEU-4':     'bleu4',
    'ACI%':       'aci',
    'ACJ-Score': 'acj_score',
}

print(f"{'Metric':<14}  {'Pearson r':>10}  {'p-value':>10}  {'Spearman ρ':>11}  {'p-value':>10}  n")
print("─" * 70)
for label, col in metrics.items():
    sub = df[['mean_rating', col]].dropna()
    n = len(sub)
    pr, pp = stats.pearsonr(sub['mean_rating'], sub[col])
    sr, sp = stats.spearmanr(sub['mean_rating'], sub[col])
    sig_p = '***' if pp < 0.001 else ('**' if pp < 0.01 else ('*' if pp < 0.05 else ''))
    sig_s = '***' if sp < 0.001 else ('**' if sp < 0.01 else ('*' if sp < 0.05 else ''))
    print(f"{label:<14}  {pr:>+.3f}{sig_p:<3}  {pp:>10.4f}  {sr:>+.3f}{sig_s:<3}  {sp:>10.4f}  {n}")

# ── 7. Optional: scatter plots ───────────────────────────────────────────────
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (label, col) in zip(axes, metrics.items()):
    sub = df[['mean_rating', col]].dropna()
    ax.scatter(sub[col], sub['mean_rating'], alpha=0.6, edgecolors='k', linewidths=0.4)
    m, b = np.polyfit(sub[col], sub['mean_rating'], 1)
    xs = np.linspace(sub[col].min(), sub[col].max(), 100)
    ax.plot(xs, m * xs + b, 'r--', lw=1.5)
    r, _ = stats.pearsonr(sub['mean_rating'], sub[col])
    ax.set_xlabel(label, fontsize=11)
    ax.set_ylabel('Mean Human Rating', fontsize=11)
    ax.set_title(f'r = {r:+.3f}', fontsize=12)
ax.figure.suptitle('Automatic Metrics vs. Human Evaluation (n=49)', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig('correlation_human_vs_metrics.pdf', bbox_inches='tight')
plt.show()
print("Saved: correlation_human_vs_metrics.pdf")

In [ ]:
# ── CELL2: ACJ-Score ablation ─────────────────────────────────────────────────
ablation = {
    'Full ACJ (α=.50, β=.40, γ=.10)': 0.50*df['bleu4_norm'] + 0.40*df['aci'] + 0.10*df['json_pct'],
    'No ACI% (α=.55, β=0, γ=.45)':    0.55*df['bleu4_norm'] + 0.00*df['aci'] + 0.45*df['json_pct'],
    'No JSON% (α=.55, β=.45, γ=0)':   0.55*df['bleu4_norm'] + 0.45*df['aci'] + 0.00*df['json_pct'],
    'BLEU-4 only':                      df['bleu4_norm'],
    'ACI% only':                        df['aci'],
    'JSON% only':                       df['json_pct'],
}

print(f"{'Configuration':<35} {'Pearson r':>10} {'p-value':>10}  n")
print("─" * 65)
for name, scores in ablation.items():
    sub = pd.DataFrame({'s': scores, 'r': df['mean_rating']}).dropna()
    r, p = stats.pearsonr(sub['r'], sub['s'])
    sig = '***' if p<0.001 else ('**' if p<0.01 else ('*' if p<0.05 else 'ns'))
    print(f"{name:<35} {r:>+.3f}      {p:>8.4f}{sig}  {len(sub)}")

In [ ]:

# ── CELL3: INSTRUCTION TYPE DIFFICULTY TAXONOMY ──────────────────────────────
print("\n" + "="*60)
print("PART 4 — Instruction type difficulty taxonomy (spaCy imperative checker)")
print("="*60)

type_rows = []
for df, name in zip(dfs, MODEL_NAMES):
    type_col = "type" if "type" in df.columns else "instruction_type"
    for itype, grp in df.groupby(type_col):
        type_rows.append({
            "Instruction Type": itype,
            "Model":            name,
            "N":                len(grp),
            "C%":               round(grp["C"].mean()   * 100, 1),
            "A%":               round(grp["A"].mean()   * 100, 1),
            "I%":               round(grp["I"].mean()   * 100, 1),
            "ACI%":             round(grp["ACI"].mean() * 100, 1),
        })

tax_df = pd.DataFrame(type_rows)

pivot = tax_df.pivot_table(
    index="Instruction Type", columns="Model", values="ACI%", aggfunc="first"
)[MODEL_NAMES]
n_map = tax_df[tax_df["Model"]=="VisualStep"].set_index("Instruction Type")["N"]
pivot.insert(0, "N (test)", n_map)

print("\nACI% by instruction type and model:")
print(pivot.to_string())

print("\n\nFull breakdown (C / A / I / ACI per type per model):")
print(tax_df[["Instruction Type","Model","N","C%","A%","I%","ACI%"]].to_string(index=False))

vs_type = (tax_df[tax_df["Model"]=="VisualStep"]
           [["Instruction Type","ACI%","N"]]
           .sort_values("ACI%"))
print("\nDifficulty ranking (VisualStep, hardest → easiest):")
for _, row in vs_type.iterrows():
    print(f"  {row['Instruction Type']:12s}  ACI%={row['ACI%']:5.1f}%  N={int(row['N'])}")

print("\n✓ Done. Use these tables in Sections 6.1 (ACJ-Score) and 6.6 (Taxonomy).")